[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/frhack/oli_ai/blob/main/notebooks/oli_ai_xor_didattica.ipynb)

# Reti neurali — perché serve la non-linearità

## Cosa abbiamo già visto

1. **Gradient descent** — un metodo generale per minimizzare una funzione di costo.
2. **Regressione lineare** — modello lineare $\hat{y} = m\,x + q$, addestrato col GD.
3. **Parametri** — i numeri che il modello impara dai dati.

## Cosa faremo oggi

Affrontiamo un problema classico — **XOR** — che il modello lineare non sa risolvere. Lo useremo come tester per scoprire, *per fallimenti successivi*, due idee fondamentali del deep learning:

1. un neurone solo **non basta** per problemi non lineari;
2. una rete più grande **ma senza attivazioni** non aiuta — la matematica la riduce a un solo neurone;
3. aggiungere una **funzione di attivazione** sblocca tutto.

Useremo Keras, che ci dà reti pronte all'uso. Sotto al cofano c'è esattamente lo stesso gradient descent della scorsa lezione.

## Setup

In [ ]:
!pip install --upgrade --no-cache-dir oli_ai > /dev/null

import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
from oli_ai.mnist_lib import show_nn_graph

np.random.seed(0)
import tensorflow as tf; tf.random.set_seed(0)

## 1. Il problema XOR

**XOR** (or esclusivo) restituisce **1 se gli input sono diversi**, 0 se sono uguali:

| x₁ | x₂ | XOR |
|---|---|---|
| 0 | 0 | 0 |
| 0 | 1 | 1 |
| 1 | 0 | 1 |
| 1 | 1 | 0 |

Disegnamo i 4 punti nel piano, colorando in nero gli output 1 e in bianco gli output 0.

In [ ]:
inputs = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype='float32')
outputs = np.array([[0],    [1],    [1],    [0]],   dtype='float32')

plt.figure(figsize=(5, 5))
for (x1, x2), y in zip(inputs, outputs.ravel()):
    plt.scatter(x1, x2, s=400, c='black' if y == 1 else 'white', edgecolor='black', linewidth=2, zorder=3)
plt.xlim(-0.3, 1.3); plt.ylim(-0.3, 1.3)
plt.xlabel('x₁'); plt.ylabel('x₂'); plt.title('I 4 punti di XOR')
plt.grid(alpha=0.3); plt.gca().set_aspect('equal'); plt.show()

Osservazione cruciale: **non esiste una retta** che separi i punti neri da quelli bianchi. I neri sono in diagonale, i bianchi sull'altra diagonale. Si dice che XOR **non è linearmente separabile**.

(Confronta con AND — i 3 zeri stanno tutti su un lato della retta $x_1 + x_2 = 1.5$ e l'unico 1 sull'altro — o con OR — separabile da $x_1 + x_2 = 0.5$.)

*(La cella sottostante è codice tecnico di grafica: definisce la funzione `mostra_decisione` che useremo per visualizzare cosa ha imparato la rete. Esegui pure.)*

In [ ]:
#@title 📊 Helper: mostra la regione di decisione del modello {display-mode: "form"}

def mostra_decisione(modello, titolo=''):
    xx, yy = np.meshgrid(np.linspace(-0.3, 1.3, 300), np.linspace(-0.3, 1.3, 300))
    grid = np.c_[xx.ravel(), yy.ravel()].astype('float32')
    Z = modello.predict(grid, verbose=0).reshape(xx.shape)
    fig, ax = plt.subplots(figsize=(6, 5))
    cs = ax.contourf(xx, yy, Z, levels=40, cmap='RdBu_r', alpha=0.7, vmin=0, vmax=1)
    ax.contour(xx, yy, Z, levels=[0.5], colors='black', linewidths=2)
    for (px, py), pl in zip(inputs, outputs.ravel()):
        ax.scatter(px, py, s=300, c='black' if pl == 1 else 'white', edgecolor='black', linewidth=2, zorder=5)
    ax.set_xlim(-0.3, 1.3); ax.set_ylim(-0.3, 1.3)
    ax.set_xlabel('x₁'); ax.set_ylabel('x₂'); ax.set_title(titolo)
    ax.set_aspect('equal')
    plt.colorbar(cs, ax=ax, label='predizione')
    plt.show()

## 2. Tentativo 1 — un singolo neurone

Un neurone con 2 ingressi calcola:

$$\hat{y} = \sigma(w_1 x_1 + w_2 x_2 + b)$$

dove $\sigma$ è la sigmoide (per ottenere un'uscita fra 0 e 1, da interpretare come 0/1). I parametri $w_1, w_2, b$ vengono appresi col GD.

Architettura `[2, 1]`: 2 input, 1 output.

In [ ]:
show_nn_graph([2, 1])

inp = Input(shape=(2,))
out = Dense(1, activation='sigmoid')(inp)
modello = Model(inp, out)

modello.compile(optimizer=Adam(learning_rate=0.05), loss='binary_crossentropy', metrics=['accuracy'])
modello.fit(inputs, outputs, epochs=1000, verbose=0)

accuratezza = modello.evaluate(inputs, outputs, verbose=0)[1]
print(f'Accuratezza: {accuratezza*100:.0f}%')
mostra_decisione(modello, titolo='Tentativo 1 — un singolo neurone')

**Si schianta.** L'accuratezza si ferma a circa il 50–75% (un caso su 4 sempre sbagliato) e nel grafico si vede chiaramente perché: il modello ha disegnato **una retta** per separare i punti, ma su XOR nessuna retta funziona.

Geometricamente, un singolo neurone con sigmoide sa solo dividere il piano in **due semipiani** lungo una retta. È poco — XOR ci chiede di più.

## 3. Tentativo 2 — una rete più grande

Idea ovvia: se un neurone non basta, aggiungiamone tanti. Mettiamo un **hidden layer** con 8 neuroni in mezzo: architettura `[2, 8, 1]`. La rete ha ora $2{\cdot}8 + 8 + 8{\cdot}1 + 1 = 33$ parametri (contro i 3 del tentativo 1).

Più capacità, dovrebbe funzionare, no?

In [ ]:
show_nn_graph([2, 8, 1])

inp = Input(shape=(2,))
h = Dense(8)(inp)                           # ← 8 neuroni hidden, NESSUNA attivazione
out = Dense(1, activation='sigmoid')(h)
modello = Model(inp, out)

modello.compile(optimizer=Adam(learning_rate=0.05), loss='binary_crossentropy', metrics=['accuracy'])
modello.fit(inputs, outputs, epochs=1000, verbose=0)

accuratezza = modello.evaluate(inputs, outputs, verbose=0)[1]
print(f'Accuratezza: {accuratezza*100:.0f}%')
mostra_decisione(modello, titolo='Tentativo 2 — [2, 8, 1] senza attivazione')

**Si schianta ancora.** Stessa accuratezza miserabile, e — sorpresa — la regione di decisione è **ancora una retta**. Avere 8 neuroni in più non ha cambiato nulla.

### Perché?

Ogni neurone hidden senza attivazione calcola una combinazione lineare dei suoi input: $h_j = \sum_i w_{ji} x_i + b_j$. L'output finale è poi un'altra combinazione lineare dei $h_j$. **Componendo due trasformazioni lineari si ottiene un'altra trasformazione lineare**. In formule, se

$$h = W_1 x + b_1, \qquad y = W_2 h + b_2$$

allora

$$y = W_2 (W_1 x + b_1) + b_2 = (W_2 W_1) x + (W_2 b_1 + b_2)$$

che è ancora $y = W' x + b'$ — equivalente a un *singolo* layer. La rete con 33 parametri sta usando, di fatto, solo 3 gradi di libertà. Aggiungere neuroni così non serve a niente.

## 4. La soluzione — funzioni di attivazione

Per uscire dal mondo lineare bisogna inserire fra un layer e l'altro una **funzione non lineare**. La più semplice e usata si chiama **ReLU** (*Rectified Linear Unit*):

$$\text{ReLU}(z) = \max(0,\, z)$$

Cioè: se l'input è positivo lo lascia stare, se è negativo lo schiaccia a zero. Banale, ma è quel "piccolo gomito" non lineare che cambia tutto.

In [ ]:
z = np.linspace(-3, 3, 100)
plt.figure(figsize=(5, 3))
plt.plot(z, np.maximum(0, z), color='#34d399', linewidth=2)
plt.axhline(0, color='gray', linewidth=0.5); plt.axvline(0, color='gray', linewidth=0.5)
plt.title('ReLU(z) = max(0, z)'); plt.grid(alpha=0.3); plt.show()

## 5. Tentativo 3 — la stessa rete, con attivazione

Riprendiamo *esattamente* la rete del tentativo 2 e cambiamo **una sola riga** — aggiungiamo `activation='relu'` al hidden layer.

In [ ]:
show_nn_graph([2, 8, 1])

inp = Input(shape=(2,))
h = Dense(8, activation='relu')(inp)        # ← cambia SOLO QUESTA riga
out = Dense(1, activation='sigmoid')(h)
modello = Model(inp, out)

modello.compile(optimizer=Adam(learning_rate=0.05), loss='binary_crossentropy', metrics=['accuracy'])
modello.fit(inputs, outputs, epochs=1000, verbose=0)

accuratezza = modello.evaluate(inputs, outputs, verbose=0)[1]
print(f'Accuratezza: {accuratezza*100:.0f}%')
mostra_decisione(modello, titolo='Tentativo 3 — [2, 8, 1] con ReLU')

**Funziona.** Accuratezza al 100%, e la regione di decisione non è più una retta: si è piegata, formando un'area che cattura esattamente la diagonale dei punti neri.

Lo stesso identico numero di neuroni e di parametri di prima. L'unica cosa cambiata è che fra un layer e l'altro c'è ora una piega non lineare. Quella piega è il deep learning.

## Riepilogo

| Architettura | Attivazione hidden | Accuratezza | Cosa ha imparato |
|---|---|---|---|
| `[2, 1]` | — | ~75% | una retta |
| `[2, 8, 1]` | nessuna | ~75% | ancora una retta |
| `[2, 8, 1]` | ReLU | 100% | una regione curva |

**Lezione**: aggiungere neuroni *non* aumenta da solo l'espressività di una rete. Ciò che la aumenta è la **non linearità** introdotta dalle funzioni di attivazione fra i layer.

Le reti che addestrano i grandi modelli moderni (GPT, riconoscimento immagini, …) sono questa stessa architettura ripetuta tante volte: tanti `Dense + activation`, in fila.

## Esercizi

1. Cambia il layout in `[2, 2, 1]` (solo 2 neuroni hidden) con `activation='relu'`. Funziona ancora? Qual è la rete *minima* che impara XOR?
2. Sostituisci `'relu'` con `'tanh'` o `'sigmoid'`. Cambia qualcosa? Convergenza più veloce o più lenta?
3. Prova `[2, 4, 4, 1]` (due hidden layer). Disegnala con `show_nn_graph` e addestrala. La regione di decisione è diversa?
4. Riprendi il tentativo 2 (rete senza attivazione hidden) e aumenta a `[2, 64, 1]`. La predizione migliora con più neuroni? Confronta con la teoria (composizione lineare = lineare).